# Train HiDDeN v2 (Google Colab)

Before running: **Runtime → Change runtime type → T4 GPU**.

Run the cells top to bottom. Checkpoints are saved to your Google Drive
(`MyDrive/thesis_checkpoints`), so if Colab disconnects just run every cell
again — training resumes from the last save.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/thesis_checkpoints'
!mkdir -p {CKPT_DIR}

In [ ]:
# Get the code (re-running pulls the newest version)
%cd /content
!test -d thesis || git clone -b claude/keen-gauss-dwwse7 https://github.com/Valeria-Balatskaya/thesis.git
%cd /content/thesis
!git pull

In [ ]:
# Training images: COCO val2017 (5,000 photos, ~1 GB), shrunk once to 288 px
!test -d /content/coco/val2017 || (mkdir -p /content/coco && cd /content/coco && wget -q http://images.cocodataset.org/zips/val2017.zip && unzip -q val2017.zip && rm val2017.zip)
!test -d /content/train_small || python scripts/prepare_training_images.py /content/coco/val2017 /content/train_small

## Train

Every 2,000 steps a validation table is printed: for each distortion, the bit
accuracy and **decodable** — the share of images whose product ID the
error-correcting code would recover. `severity` ramps from 0 to 1 by step
15,000, so accuracy dips during the ramp and then recovers; judge the model by
the tables after step 15,000.

`--target-psnr` is the image-quality budget (40 dB default). Higher = less
visible but less robust.

In [ ]:
import os
resume = f'{CKPT_DIR}/hidden_v2_latest.pt'
resume_flag = f'--resume {resume}' if os.path.exists(resume) else ''
!python -u -m src.training.train_v2 --data /content/train_small --out {CKPT_DIR} --steps 40000 --batch-size 32 --target-psnr 40 {resume_flag}

## Download the result

Put the downloaded file in the project's `checkpoints/` folder on your Mac,
then run `python tests/test_hidden_v2_eval.py`.

In [ ]:
from google.colab import files
files.download(f'{CKPT_DIR}/hidden_v2_best.pt')